# Training a normalizing flow for H$_2$ (orbital-free DFT)

This example mirrors the 1D `LiH_1D_NNX` tutorial, but for **H$_2$ in 3D** using the
`off` package. The electron density is represented by a **continuous normalizing flow
(CNF)** and the ground state is obtained by **minimizing the OF-DFT energy** with a
Monte-Carlo estimator.

The flow evolves the augmented ODE
$$
\partial_t \begin{bmatrix} \mathbf{z}(t) \\ \log\rho_\phi \\ \nabla\log\rho_\phi \end{bmatrix}
= \begin{bmatrix} g_\phi(\mathbf{z},t) \\ -\nabla\!\cdot g_\phi \\ -(\nabla g_\phi)^\top \nabla\log\rho_\phi - \nabla(\nabla\!\cdot g_\phi) \end{bmatrix},
$$
which yields both $\rho_\phi$ and its score $\nabla\log\rho_\phi$ — the inputs the density
functionals consume. The energy is the Monte-Carlo average over samples $\mathbf{x}_i\sim\rho_\phi$,
$$
E_{\rm gs} \approx \frac{1}{N}\sum_i \big(T + V_{\rm ne} + V_{\rm H} + E_{\rm xc}\big)(\mathbf{x}_i).
$$


In [1]:
import time
import jax
import jax.numpy as jnp
import jax.random as jrnd
import pandas as pd
import matplotlib.pyplot as plt

import off
from off.train import (setup_molecule, setup_model, setup_optimizer,
                       setup_ema, step, create_loss_function, log_metrics)
from off.utils import get_solver, batch_generator
from off.promolecular.promolecular_dist import ProMolecularDensity

jax.config.update("jax_enable_x64", True)
print("off version:", off.__version__)

ImportError: cannot import name 'prng' from 'jax._src' (/opt/homebrew/Caskroom/miniconda/base/envs/off/lib/python3.11/site-packages/jax/_src/__init__.py)

## 1. The H$_2$ system

`setup_molecule` returns the electron count, atom labels, charges, nuclear coordinates
(in Bohr), and the `mol` dict that the nuclear/Hartree terms consume.

In [2]:
mol_name    = "H2"
bond_length = 1.4            # Bohr (H2 equilibrium ~ 1.4 a0)

Ne, atoms, z, coords, mol = setup_molecule(mol_name, bond_length)
print("Ne =", Ne, "| atoms =", atoms)
print("coords (Bohr):\n", coords)

NameError: name 'setup_molecule' is not defined

## 2. Choose the energy functional

We use **TF-$\lambda$W** kinetic ($\lambda=1/5$), **LDA + B88** exchange, the nuclear
attraction, and the **all-pairs Hartree** estimator. `create_loss_function` assembles
the `EnergyFunctional` and returns the Monte-Carlo loss
`grad_loss(model, batch, solver, Ne, mol)`.

In [ ]:
epochs       = 500          # reduce (e.g. 100) for a quick smoke test
batch_size   = 512
hidden_layer = 64
lr           = 3e-4

key = jrnd.PRNGKey(0)
_, key = jrnd.split(key)

flow_model              = setup_model(coords, z, hidden_layer, key)
solver                  = get_solver("dopri8")
optimizer, opt_state    = setup_optimizer(flow_model, epochs, lr, scheduler_type="mix")
energies_ema, ema_state = setup_ema()

prior   = ProMolecularDensity(z.ravel(), coords)        # promolecular base distribution
batches = batch_generator(key, batch_size, prior)

grad_loss = create_loss_function(
    kinetic_name="tf_w", lam=1/5,
    exchange_name="lda_b88_x", correlation_name="none",
    hartree_name="coulomb",    external_name="np", core_correction_name="none",
)

## 3. Training loop

Each step samples a batch from the base distribution, flows it forward, evaluates the
energy, and updates the flow. We track an exponential moving average (EMA) of the noisy
Monte-Carlo energy. (This is exactly what `off.training` / the `off-train` CLI run,
written out so you can see every piece.)

In [ ]:
df = pd.DataFrame()
for itr in range(epochs + 1):
    t0 = time.time()
    batch = next(batches)
    loss, flow_model, opt_state = step(
        flow_model, batch, optimizer, opt_state, grad_loss, solver, Ne, mol)
    loss_epoch, losses = loss
    e_ema, ema_state = energies_ema.update(losses, ema_state)
    _, r_ema = log_metrics(itr, loss_epoch, losses, e_ema, time.time() - t0)
    df = pd.concat([df, pd.DataFrame([r_ema])], ignore_index=True)
    if itr % 20 == 0:
        print(f"epoch {itr:4d}   E_ema = {r_ema['E']:+.6f} Ha")

## 4. Energy convergence

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(df["epoch"], df["E"], lw=2)
plt.xlabel("epoch"); plt.ylabel(r"$E_{\rm EMA}$ [Ha]")
plt.title(r"H$_2$ OF-DFT energy (EMA)"); plt.grid(alpha=0.3)
plt.show()
print("final EMA energy:", float(df["E"].iloc[-1]), "Ha")

## 5. Density along the bond axis

Flow points on the H–H line back to the base distribution and read off
$\rho(\mathbf{x}) = N_e\,\rho_\phi(\mathbf{x})$.

In [ ]:
from off.ode_solver import rev_ode, fwd_ode

a, b = coords[0], coords[1]
t    = jnp.linspace(-1.0, 2.0, 200)
pts  = a[None, :] + t[:, None] * (b - a)[None, :]          # points along the bond
n    = pts.shape[0]

state_rev = jnp.concatenate([pts, jnp.zeros((n, 1)), jnp.zeros((n, 3))], axis=1)
zb, _ = rev_ode(flow_model, state_rev, solver)
lp0, sc0 = prior.log_prob(zb), prior.score(zb)
_, lpt, _ = fwd_ode(flow_model, jnp.concatenate([zb, lp0, sc0], axis=1), solver)
rho = Ne * jnp.exp(lpt).ravel()

dist = t * float(jnp.linalg.norm(b - a))
plt.figure(figsize=(6, 4))
plt.plot(dist, rho, lw=2)
plt.axvline(0.0, ls="--", c="k", alpha=0.5)
plt.axvline(float(jnp.linalg.norm(b - a)), ls="--", c="k", alpha=0.5)
plt.xlabel("distance along bond [Bohr]"); plt.ylabel(r"$\rho$")
plt.title(r"H$_2$ density along the bond"); plt.grid(alpha=0.3)
plt.show()

## 6. Save the trained flow

Save the checkpoint and `job_params.json` in the same layout the `off-train` CLI
produces, so the **loading** and **quadrature** notebooks can pick this run up via
`off.grid_energy_from_checkpoint(run_dir)`.

In [ ]:
import os, json
import equinox as eqx

run_dir = "h2_run/bl_1.40"
os.makedirs(f"{run_dir}/Checkpoints", exist_ok=True)
eqx.tree_serialise_leaves(f"{run_dir}/Checkpoints/checkpoint_{epochs}.eqx", flow_model)

job_params = {
    "model": "cnf", "mol_name": mol_name, "bond_length": bond_length,
    "epochs": epochs, "batch_size": batch_size, "hidden_layer": hidden_layer, "lr": lr,
    "kinetic": "tf_w", "lam": 1/5, "external": "np", "hartree": "coulomb",
    "exchange": "lda_b88_x", "correlation": "none", "core_correction": "none",
    "scheduler": "mix", "solver": "dopri8", "prior": "promolecular",
}
with open(f"{run_dir}/job_params.json", "w") as f:
    json.dump(job_params, f, indent=4)
df.to_csv(f"{run_dir}/training_metrics_ema.csv", index=False)
print("saved ->", run_dir)

> **Production shortcut.** The same run is one command:
> ```bash
> off-train --mol_name H2 --bond_length 1.4 --kin tf_w --lam 1/5 \
>           --x lda_b88_x --hart coulomb --prior promolecular \
>           --solver dopri8 --epochs 500 --bs 512
> ```
> or `off.training(...)` in Python — both add checkpointing and CSV logging
> automatically and write to `Results/H2/<method>/bl_1.40/`.